#  🍄 SUSTAIN 🍄
### A Practical Example
Inspired by the famous edible-mushroom problem, we aim to solve it using technology from the 80s! I this notebook
we are trying to classify the mushrooms based on their features.

Big shootout for the creators of the dataset: https://archive.ics.uci.edu/dataset/73/mushroom !

### (1) Preprocessing
We will load the dataset and then encode a few chosen features of mushrooms as vectors.


In [34]:
import pandas as pd
import os

In [35]:
print(f"working dir: {os.getcwd()}")

working dir: C:\Users\kalus\PycharmProjects\SUSTAIN-replication


In [36]:
df = pd.read_csv("mushrooms.csv", sep=",", encoding="utf-8")
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8124 entries, 0 to 8123
Data columns (total 23 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   class                     8124 non-null   str  
 1   cap-shape                 8124 non-null   str  
 2   cap-surface               8124 non-null   str  
 3   cap-color                 8124 non-null   str  
 4   bruises                   8124 non-null   str  
 5   odor                      8124 non-null   str  
 6   gill-attachment           8124 non-null   str  
 7   gill-spacing              8124 non-null   str  
 8   gill-size                 8124 non-null   str  
 9   gill-color                8124 non-null   str  
 10  stalk-shape               8124 non-null   str  
 11  stalk-root                8124 non-null   str  
 12  stalk-surface-above-ring  8124 non-null   str  
 13  stalk-surface-below-ring  8124 non-null   str  
 14  stalk-color-above-ring    8124 non-null   str  
 15

In [37]:
# the most important variables are class
df.sample(4)

,class,cap-shape,cap-surface,cap-color,bruises,odor,gill-attachment,gill-spacing,gill-size,gill-color,...,stalk-surface-below-ring,stalk-color-above-ring,stalk-color-below-ring,veil-type,veil-color,ring-number,ring-type,spore-print-color,population,habitat
7659,e,k,f,w,f,n,f,w,b,w,...,k,w,w,p,w,t,p,w,s,g
144,e,x,y,y,t,a,f,c,b,k,...,s,w,w,p,w,o,p,k,n,g
4866,p,f,f,g,f,f,f,c,b,p,...,k,p,p,p,w,o,l,h,v,p
6482,p,f,s,e,f,s,f,c,n,b,...,s,p,w,p,w,o,e,w,v,d


What we have to do now is to transform the output into the list readable for SUSTAIN.

In [38]:
X = df[['cap-shape','cap-surface', 'cap-color', 'bruises', 'odor']]
y = df["class"]

# for tests
X = X.head(15)
y = y.head()

# from sklearn.model_selection import train_test_split
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.2, random_state=0, stratify=y)

In [47]:
from sklearn.preprocessing import OrdinalEncoder

split_index = int(len(df) * 0.70)
data_train = df.iloc[:split_index]
data_test = df.iloc[split_index:]


def encode_stimuli(data: pd.DataFrame):

    # convert to a list
    stimuli = []
    for row in data.itertuples(index=False):
        row_list = list(row)
        stimuli.append(row_list)

    # the list consts of string, we need numbers
    encoder = OrdinalEncoder()
    encoded_stimuli = encoder.fit_transform(stimuli).astype(int).tolist()

    # compute the size of dimensions
    dim_sizes = [len(categories) for categories in encoder.categories_]
    print(f"dim_sizes: {dim_sizes}")

    return encoded_stimuli, dim_sizes


train_encoded_stimuli, _ = encode_stimuli(data_train)
test_encoded_stimuli, dim_sizes = encode_stimuli(data_test)

sustain_run = [train_encoded_stimuli, test_encoded_stimuli]
print(len(test_encoded_stimuli))

dim_sizes: [2, 6, 4, 10, 2, 8, 1, 2, 2, 10, 2, 5, 3, 4, 6, 7, 1, 1, 2, 4, 6, 6, 7]
dim_sizes: [2, 5, 4, 10, 2, 5, 2, 2, 2, 11, 2, 3, 4, 4, 8, 8, 1, 4, 3, 5, 7, 5, 7]
2438


In [48]:
from sustain import SUSTAIN

model = SUSTAIN(r=2.844642,
                beta=2.386305,
                d=12.0,
                eta=0.09361126,
                supervised=True,
                queried_dim=0)

model.reset(dim_sizes=dim_sizes)  # dim0=feature, dim1=category


index = []
response = []
probability = []
correct = []
n_clusters = []
winner = []
recruited = []

for j, run in enumerate(sustain_run):
    for i, stim in enumerate(run):
        res = model.present_stimulus(stim, queried_dim=0)

        if j ==1:
            index.append(i)
            response.append(res['response'])
            probability.append(res['prob'])
            correct.append(res['correct'])
            n_clusters.append(res['n_clusters'])
            winner.append(res['winner'])
            recruited.append(res['recruited'])


results = pd.DataFrame({
    "index": index,
    "response": response,
    "probability": probability,
    "correct": correct,
    "n_clusters": n_clusters,
    "winner": winner,
    "recruited": recruited
})

results.set_index("index", inplace=True)


In [49]:
results

,response,probability,correct,n_clusters,winner,recruited
index,,,,,,
0,0,"[0.5000899418231878, 0.4999100581768123]",True,36,21,False
1,0,"[0.5000273829804044, 0.49997261701959556]",True,36,25,False
2,1,"[0.49996509510074666, 0.5000349048992534]",True,36,19,False
3,1,"[0.49777624022132083, 0.5022237597786791]",True,36,15,False
4,1,"[0.49996455193212636, 0.5000354480678737]",True,36,20,False
...,...,...,...,...,...,...
2433,0,"[0.5000049538374923, 0.49999504616250773]",True,70,61,False
2434,0,"[0.5000041232851458, 0.49999587671485424]",True,70,61,False
2435,0,"[0.5000050012511897, 0.4999949987488102]",True,70,61,False


In [50]:
count_true = (results["correct"] == True).sum()
count_false = (results["correct"] == False).sum()
accuracy = count_true / (count_false + count_true)
print(accuracy)

0.9860541427399507


In [51]:
count_false

np.int64(34)